# Z integrity metric

ovrlpy gives each 1 µm pixel a z integrity (its vertical signal integrity). Low z integrity (< 0.7) means the transcripts above and below differ, so the signal mixes cells stacked in z.

For each method and sample, `compute_z_integrity` pools all pixels inside the method's post-QC cells, keeps pixels with signal > 2 and reports the % with low z integrity (`low_z_integrity_pct`, lower is better). Pooling makes the score independent of how a region is split into cells, so oversegmentation is not penalised. Coverage (% of signal pixels inside cells) is reported beside it, because methods that cover less tissue can avoid or hit regions with low z integrity.

Results: `metrics/<cohort>/z_integrity/z_integrity_all.csv` (and `_vascular_celltypes.csv` for the vascular subset), one row per sample and threshold pair (signal 1/2/3, z integrity 0.6/0.7/0.8), plus `all_samples` rows with the mean over samples. Needs the z integrity maps from `scripts/seg_postprocessing/ovrlpy_z_integrity_map.py`.

## Compute

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

from cellseg_benchmark import BASE_PATH, _constants
from cellseg_benchmark._constants import method_colors
from cellseg_benchmark.metrics import (
    clean_method_name,
    compute_metric_for_all_methods,
    compute_z_integrity,
)

base = Path(BASE_PATH)
plot_path = base / "metrics" / "z_integrity_plots"
plot_path.mkdir(parents=True, exist_ok=True)
cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]

dfs = []
for cohort in cohorts:
    methods = [m.name for m in (base / "analysis" / cohort).iterdir() if not m.name.startswith(tuple(_constants.methods_3D))]
    res = compute_metric_for_all_methods(compute_z_integrity, cohort, "z_integrity/z_integrity_all.csv", methods=methods, obs_only=True)
    dfs.append(res.assign(cohort=cohort))
    if cohort == "aging":
        compute_metric_for_all_methods(compute_z_integrity, cohort, "z_integrity/z_integrity_vascular_celltypes.csv",
                                       methods=methods, adata_name="adata_vascular_subset", obs_only=True)
df = pd.concat(dfs).query("sample != 'all_samples' and method != 'Negative_Control_Visium'")
main = df.query("signal_threshold == 2 and z_integrity_threshold == 0.7")

## 1. Low z integrity share per cohort

Dots are samples, ticks are medians. Coverage on the right.

In [ ]:
n = main["method"].nunique()
fig, axs = plt.subplots(len(cohorts), 2, figsize=(9, 0.25 * n * len(cohorts) + 1), gridspec_kw={"width_ratios": [3, 1]})
for i, cohort in enumerate(cohorts):
    d = main[main["cohort"] == cohort].assign(label=lambda x: x["method"].map(clean_method_name))
    order = d.groupby("label")["low_z_integrity_pct"].median().sort_values().index
    pal = dict(zip(d["label"], d["method"].map(lambda m: method_colors.get(m, "0.5"))))
    for ax, col, xlabel in [(axs[i, 0], "low_z_integrity_pct", "Low z integrity signal in cells (%)"), (axs[i, 1], "coverage_pct", "Coverage (%)")]:
        sns.stripplot(d, x=col, y="label", hue="label", order=order, palette=pal, size=3, jitter=0.2, legend=False, ax=ax)
        sns.boxplot(d, x=col, y="label", order=order, showbox=False, showcaps=False, whiskerprops={"visible": False},
                    showfliers=False, medianprops={"color": "k", "lw": 1.5}, ax=ax)
        ax.set(xlabel=xlabel, ylabel="", title=cohort if col == "low_z_integrity_pct" else "")
        ax.grid(axis="x", alpha=0.3)
        if col == "coverage_pct":
            ax.set(xlim=(0, 100), yticklabels=[])
plt.tight_layout()
plt.savefig(plot_path / "z_integrity_per_cohort.png", dpi=200, bbox_inches="tight")
plt.show()

## Ranking

Rank of each method per cohort (mean over samples, 1 = lowest share), ordered by mean rank across cohorts.

In [ ]:
rank = main.groupby(["cohort", "method"])["low_z_integrity_pct"].mean().groupby(level=0).rank().reset_index(name="rank")
rank["label"] = rank["method"].map(clean_method_name)
order = rank.groupby("label")["rank"].mean().sort_values().index
fig, ax = plt.subplots(figsize=(5, 0.25 * len(order) + 1))
sns.stripplot(rank, x="rank", y="label", hue="cohort", order=order, size=5, jitter=False, ax=ax)
sns.boxplot(rank, x="rank", y="label", order=order, showbox=False, showcaps=False, whiskerprops={"visible": False},
            showfliers=False, showmeans=True, meanline=True, meanprops={"color": "k", "lw": 1.5, "ls": "-"}, medianprops={"visible": False}, ax=ax)
ax.set(xlabel="Rank (1 = least low z integrity signal)", ylabel="")
ax.legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="x", alpha=0.3)
plt.savefig(plot_path / "z_integrity_ranking.png", dpi=200, bbox_inches="tight")
plt.show()

## 2. Low z integrity share vs coverage

Methods covering almost all tissue sit at the tissue average. All spread between methods comes from methods with lower coverage.

In [ ]:
fig, axs = plt.subplots(1, len(cohorts), figsize=(5 * len(cohorts), 4.5))
for ax, cohort in zip(axs, cohorts):
    m = main[main["cohort"] == cohort].groupby("method")[["coverage_pct", "low_z_integrity_pct"]].mean().reset_index()
    ax.scatter(m["coverage_pct"], m["low_z_integrity_pct"], c=m["method"].map(lambda x: method_colors.get(x, "0.5")), s=30, edgecolor="k", lw=0.3)
    for _, r in m.nlargest(3, "low_z_integrity_pct").iterrows():
        ax.annotate(clean_method_name(r["method"]), (r["coverage_pct"], r["low_z_integrity_pct"]), fontsize=7)
    rho = m[["coverage_pct", "low_z_integrity_pct"]].corr("spearman").iloc[0, 1]
    ax.set(xlabel="Coverage (%)", ylabel="Low z integrity signal in cells (%)", title=f"{cohort} (rho {rho:.2f})")
plt.tight_layout()
plt.savefig(plot_path / "z_integrity_vs_coverage.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Sensitivity

Spearman correlation of the method order (mean over samples) at each threshold pair with the default (signal 2, z integrity 0.7), and how well samples within a cohort agree (mean pairwise Spearman between samples).

In [ ]:
order_at = df.groupby(["cohort", "signal_threshold", "z_integrity_threshold", "method"])["low_z_integrity_pct"].mean().unstack("method")
default = order_at.xs((2, 0.7), level=("signal_threshold", "z_integrity_threshold"))
sens = pd.DataFrame({c: order_at.loc[c].T.corrwith(default.loc[c], method="spearman") for c in cohorts}).round(2)

def sample_agreement(d):
    r = d.pivot(index="method", columns="sample", values="low_z_integrity_pct").corr("spearman").to_numpy()
    return r[np.triu_indices_from(r, 1)].mean()

print(main.groupby("cohort").apply(sample_agreement).round(2).to_string())
sens

## 4. Do cells with low z integrity carry mixed signal?

Validity check: cells with at least half of their signal pixels at low z integrity should express more markers of other cell types. For a few methods and samples, marker F1 and the marker false positive rate (cells of other types expressing the marker) are compared between these cells and the rest, within quartiles of transcript counts so that depth does not drive the difference. Positive FPR differences and negative F1 differences support the metric.

In [ ]:
from cellseg_benchmark.metrics import (
    compute_marker_F1_score,
    low_z_integrity_per_cell,
    read_adata,
)

check_methods = ["Cellpose_1_DAPI_PolyT", "Baysor_2D_Cellpose_1_DAPI_PolyT_0.8", "vpt_2D_DAPI_PolyT", "Negative_Control_Voronoi"]
n_samples = 2

checks = []
for cohort in cohorts:
    for method in check_methods:
        adata = read_adata(cohort, method)
        if adata is None:
            continue
        adata = adata[adata.obs["sample"].isin(sorted(adata.obs["sample"].unique())[:n_samples])].copy()
        counts = adata.obs["n_counts"] if "n_counts" in adata.obs else pd.Series(np.asarray(adata.X.sum(1)).ravel(), index=adata.obs_names)
        adata.obs["depth"] = counts.groupby(adata.obs["sample"]).transform(lambda x: pd.qcut(x.rank(method="first"), 4, labels=False))
        adata.obs["low_z_integrity_cell"] = low_z_integrity_per_cell(adata, method, base).to_numpy() >= 0.5
        for (depth, low), obs in adata.obs.groupby(["depth", "low_z_integrity_cell"]):
            f1 = compute_marker_F1_score(adata[obs.index].copy(), celltype_name="cell_type_revised")
            f1["fpr"] = f1["fp"] / (len(obs) - f1["tp"] - f1["fn"]).where(lambda n: n > 0)
            checks.append(f1.assign(cohort=cohort, method=method, depth=depth, low_z_integrity_cell=low))
        del adata
checks = pd.concat(checks)

In [ ]:
strata = checks.groupby(["cohort", "method", "sample", "depth", "low_z_integrity_cell"])[["f1_score", "precision", "recall", "fpr"]].mean().unstack("low_z_integrity_cell")
diff = (strata.xs(True, axis=1, level=1) - strata.xs(False, axis=1, level=1)).dropna().reset_index()
cols = ["f1_score", "precision", "recall", "fpr"]
print(pd.concat({"mean diff": diff.groupby("cohort")[cols].mean(), "share > 0": (diff[cols] > 0).groupby(diff["cohort"]).mean()}, axis=1).round(3).to_string())

fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, col, label in [(axs[0], "f1_score", "Marker F1"), (axs[1], "fpr", "Marker false positive rate")]:
    sns.stripplot(diff, x=col, y="cohort", hue="method", dodge=True, size=4, ax=ax, legend=ax is axs[1])
    ax.axvline(0, color="k", lw=0.8)
    ax.set(xlabel=f"{label}\nlow z integrity cells minus rest", ylabel="")
axs[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=7)
plt.tight_layout()
plt.savefig(plot_path / "z_integrity_marker_check.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Cell types: low z integrity per gene (segmentation free)

For each gene, the % of its transcripts on signal pixels that have low z integrity, grouped by the cell supertype the gene marks in the 500-gene panel (`_markers.cell_supertypes_merfish_panel_500_mouse_brain`, genes marking one supertype only). Dots are genes, ticks are medians. Abundance check: if highly expressed genes simply sit in dense tissue, the share would rise with transcript count (rho in the titles), and supertype differences would vanish after correcting for it (residual table).

In [ ]:
from cellseg_benchmark._markers import (
    cell_supertypes_merfish_panel_500_mouse_brain as supertypes,
)
from cellseg_benchmark.metrics import low_z_integrity_per_gene

genes = []
for cohort in cohorts:
    f = base / "metrics" / cohort / "z_integrity" / "low_z_integrity_per_gene.csv"
    if not f.exists():
        samples = sorted(df.loc[df["cohort"] == cohort, "sample"].unique())
        pd.concat(low_z_integrity_per_gene(s, base).assign(sample=s) for s in samples).to_csv(f, index=False)
    genes.append(pd.read_csv(f).assign(cohort=cohort))
genes = pd.concat(genes)
genes = genes[~genes["gene"].str.startswith("Blank")].groupby(["cohort", "gene"])[["n", "n_low"]].sum()
genes["low_pct"] = 100 * genes["n_low"] / genes["n"]

owner = pd.Series({g: t for t, gs in supertypes.items() for g in gs})
owner = owner[~owner.index.duplicated(keep=False)]
genes["type"] = genes.index.get_level_values("gene").map(owner)
genes.groupby(["cohort", "type"])["low_pct"].agg(["median", "size"]).unstack(0).round(1)

In [ ]:
from scipy.stats import spearmanr

fig, axs = plt.subplots(1, len(cohorts), figsize=(4 * len(cohorts), 3.5), sharey=False)
resid = {}
for ax, cohort in zip(axs, cohorts):
    g = genes.loc[cohort].dropna(subset=["type"])
    order = g.groupby("type")["low_pct"].median().sort_values().index
    sns.stripplot(g, x="low_pct", y="type", order=order, color="0.6", size=2.5, jitter=0.25, ax=ax)
    sns.boxplot(g, x="low_pct", y="type", order=order, showbox=False, showcaps=False, whiskerprops={"visible": False},
                showfliers=False, medianprops={"color": "k", "lw": 1.5}, ax=ax)
    a = genes.loc[cohort]
    fit = np.polyfit(np.log10(a["n"]), a["low_pct"], 1)
    resid[cohort] = (a["low_pct"] - np.polyval(fit, np.log10(a["n"]))).groupby(a["type"]).median()
    ax.set(xlabel="Transcripts with low z integrity (%)", ylabel="", title=f"{cohort} (rho vs abundance {spearmanr(np.log10(a['n']), a['low_pct']).statistic:.2f})")
plt.tight_layout()
plt.savefig(plot_path / "z_integrity_per_supertype.png", dpi=200, bbox_inches="tight")
plt.show()
pd.DataFrame(resid).round(1)

## 6. Brain regions (segmentation free)

Share of signal pixels with low z integrity per Banksy brain region (`misc/brain_regions/<cohort>_brain_regions.parquet`, only cohorts with region labels). Dots are samples, ticks are medians; mean signal per region shows whether differences follow transcript density. Right: whole-tissue share by age, coloured by slide.

In [ ]:
import geopandas as gpd
from rasterio.features import rasterize

from cellseg_benchmark.metrics import load_z_integrity_maps

meta = yaml.safe_load(open(base / "misc" / "sample_metadata.yaml"))
rows = []
for cohort in cohorts:
    f = base / "misc" / "brain_regions" / f"{cohort}_brain_regions.parquet"
    if not f.exists():
        continue
    regions = gpd.read_parquet(f)
    labels = sorted(regions["label_broad"].unique())
    for s, reg in regions.groupby("sample"):
        if not (base / "samples" / s / "vertical_doublets_ovrlpy_output.npz").exists():
            continue
        integrity, signal, shift = load_z_integrity_maps(s, base)
        lab = rasterize(zip(reg.geometry.translate(*shift), reg["label_broad"].map(labels.index) + 1), out_shape=integrity.shape, dtype=np.int16)
        on = signal > 2
        for i, name in enumerate(["all"] + labels):
            m = on if name == "all" else on & (lab == i)
            if m.any():
                rows.append({"cohort": cohort, "sample": s, "slide": s.rsplit("_", 1)[0], "age_months": meta[s].get("age_months"), "region": name,
                             "low_pct": 100 * (integrity[m] < 0.7).mean(), "mean_signal": signal[m].mean()})
regions_df = pd.DataFrame(rows)
regions_df.groupby(["cohort", "region"])[["low_pct", "mean_signal"]].median().round(2)

In [ ]:
for cohort, d in regions_df.groupby("cohort"):
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [3, 2]})
    r = d[d["region"] != "all"]
    order = r.groupby("region")["low_pct"].median().sort_values().index
    sns.stripplot(r, x="low_pct", y="region", order=order, color="0.6", size=3, jitter=0.2, ax=axs[0])
    sns.boxplot(r, x="low_pct", y="region", order=order, showbox=False, showcaps=False, whiskerprops={"visible": False},
                showfliers=False, medianprops={"color": "k", "lw": 1.5}, ax=axs[0])
    axs[0].set(xlabel="Signal with low z integrity (%)", ylabel="", title=f"{cohort}: brain regions")
    t = d[d["region"] == "all"]
    sns.scatterplot(t, x="age_months", y="low_pct", hue="slide", s=40, ax=axs[1])
    rho = t["age_months"].corr(t["low_pct"], method="spearman")
    axs[1].set(xlabel="Age (months)", ylabel="Signal with low z integrity (%)", title=f"Whole tissue by age (rho {rho:.2f})")
    axs[1].legend(fontsize=7, frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(plot_path / f"z_integrity_regions_{cohort}.png", dpi=200, bbox_inches="tight")
    plt.show()